In [ ]:
import os
import pathlib
import subprocess
import time
import platform

root = pathlib.Path.cwd().resolve()
if not (root / 'Sockets' / 'key-value-server').exists():
    root = root.parent

# Detect OS to configure paths and compiler flags
is_windows = platform.system() == 'Windows'

if is_windows:
    # Add MSYS2 runtime directory to PATH so the DLLs (msys-2.0.dll) can be resolved
    os.environ['PATH'] = 'C:\\msys64\\usr\\bin;' + os.environ.get('PATH', '')
    exe_suffix = '.exe'
    build_dir = 'build-msys'
    cmake_cmd = 'C:\\msys64\\usr\\bin\\cmake.exe'
    generator_args = ['-G', 'Unix Makefiles']
else:
    exe_suffix = ''
    build_dir = 'build'
    cmake_cmd = 'cmake'
    generator_args = []

server_bin = root / 'Sockets' / 'key-value-server' / build_dir / f'key_value_server{exe_suffix}'
client_bin = root / 'Sockets' / 'echo-client' / build_dir / f'echo_client{exe_suffix}'

# Automatically compile the binaries if they don't exist yet
if not server_bin.exists():
    print("Compiling Key-Value Server...")
    subprocess.run([cmake_cmd] + generator_args + ['-S', 'Sockets/key-value-server', '-B', f'Sockets/key-value-server/{build_dir}'], check=True, cwd=root)
    subprocess.run([cmake_cmd, '--build', f'Sockets/key-value-server/{build_dir}'], check=True, cwd=root)

if not client_bin.exists():
    print("Compiling Echo Client...")
    subprocess.run([cmake_cmd] + generator_args + ['-S', 'Sockets/echo-client', '-B', f'Sockets/echo-client/{build_dir}'], check=True, cwd=root)
    subprocess.run([cmake_cmd, '--build', f'Sockets/echo-client/{build_dir}'], check=True, cwd=root)

commands = [
    'EXIST foo',
    'CLEAR',
    'KEYS',
    'EXIST unknown',
    'KEYS',
    'QUIT',
]

print('Demo commands:')
for command in commands:
    print('  -', command)

server_proc = subprocess.Popen([str(server_bin)], cwd=root, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)

try:
    for _ in range(20):
        if server_proc.poll() is not None:
            raise RuntimeError('Server exited before accepting a client.')
        time.sleep(0.25)
        break

    result = subprocess.run(
        [str(client_bin)],
        input='\n'.join(commands) + '\n',
        capture_output=True,
        text=True,
        timeout=10,
        cwd=root,
    )

    print('\nClient output:')
    print(result.stdout)
    if result.stderr:
        print('stderr:', result.stderr)
finally:
    server_proc.terminate()
    try:
        server_proc.wait(timeout=5)
    except subprocess.TimeoutExpired:
        server_proc.kill()
        server_proc.wait()
